## WEEK -04

## SECTION 1: SETUP


In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as _sum
import os

spark = SparkSession.builder.appName("LogiScale_Pipeline").getOrCreate()

print("Pipeline Started...")

Pipeline Started...


## SECTION 2: LOAD WEEK 1 DATA

In [25]:
CSV_PATH = r"C:\Users\Pravith Kumar J\Downloads\final_output.csv"

df = spark.read.csv(CSV_PATH, header=True, inferSchema=True)

print("Data Loaded")

Data Loaded


## SECTION 3: DATA CLEANING

In [27]:
df = df.filter(col("Delivery Status") != "Shipping canceled")
df = df.dropna()

print("Data Cleaned")

Data Cleaned


## SECTION 4: FEATURE ENGINEERING

In [29]:
df = df.withColumn("delay_days", col("Days for shipping (real)") - col("Days for shipment (scheduled)"))

## SECTION 5: AGGREGATIONS (WEEK 2 LOGIC)

In [31]:
from pyspark.sql.functions import year, month, to_date, col

df = df.withColumn(
    "order_date",
    to_date(col("order date (DateOrders)"))
)

df = df.withColumn("order_year", year(col("order_date")))
df = df.withColumn("order_month", month(col("order_date")))

In [32]:
route_efficiency = df.groupBy("Order Region").agg(
    avg("delay_days").alias("avg_delay"),
    count("Order Id").alias("total_shipments")
)

monthly_trends = df.groupBy("order_year", "order_month").agg(
    _sum("Order Item Quantity").alias("monthly_units"),
    _sum("Sales").alias("monthly_revenue")
)

## SECTION 6: CONVERT TO PANDAS

In [34]:
route_pd = route_efficiency.toPandas()
monthly_pd = monthly_trends.toPandas()

## SECTION 7: FORECASTING

In [36]:
monthly_pd = monthly_pd.sort_values(["order_year", "order_month"])

monthly_pd["forecast"] = monthly_pd["monthly_units"].rolling(window=3).mean()

## SECTION 8: EXPORT FINAL FILES

In [38]:
OUTPUT_PATH = r"C:\Users\Pravith Kumar J\OneDrive\Documents"

route_file = os.path.join(OUTPUT_PATH, "final_route_efficiency.csv")
monthly_file = os.path.join(OUTPUT_PATH, "final_monthly_trends.csv")

route_pd.to_csv(route_file, index=False)
monthly_pd.to_csv(monthly_file, index=False)

print("Files Exported Successfully")

Files Exported Successfully


## SECTION 9: EXECUTIVE SUMMARY (AUTO INSIGHTS)

In [40]:
if not route_pd.empty:
    max_delay_region = route_pd.loc[route_pd["avg_delay"].idxmax()]
    min_delay_region = route_pd.loc[route_pd["avg_delay"].idxmin()]

    print("\n--- EXECUTIVE SUMMARY ---")
    print(f"Highest Delay Region: {max_delay_region['Order Region']} ({max_delay_region['avg_delay']:.2f} days)")
    print(f"Best Performing Region: {min_delay_region['Order Region']} ({min_delay_region['avg_delay']:.2f} days)")
    print(f"Total Shipments: {route_pd['total_shipments'].sum()}")
else:
    print("No data available for analysis")

No data available for analysis


In [41]:
print("Pipeline Completed Successfully")

Pipeline Completed Successfully
